# Experiment 1 — Domain 2 (Graphs): Phase 1 Dataset Generation

Generates the **300-graph, 8-property** graph dataset for Experiment 1
(`serialization_experiment_1.pdf`, Section 4).

**Revision note (this version):** `is_bipartite` and `is_planar` are each
generated as an **exact 50/50 split, in every tier**, instead of following
the PDF's literal "aim for approximately 25-35%" target -- and **all 5
Table-7 families are exactly equal in size** (20/tier each), which the PDF's
own "aim for 18-22" guidance doesn't quite reach either. Both come from the
same design: every family covers whichever (`is_bipartite`, `is_planar`)
quadrants make sense for it, so `is_bipartite`/`is_planar` truth doesn't
belong to one or two "designed" families -- an `erdos_renyi`,
`barabasi_albert`, or `watts_strogatz` graph can genuinely be bipartite,
planar, both, or neither, each in its own family-characteristic way (see
cell 2/4). This closes two issues an earlier build surfaced in Phase 3
evaluation: (1) a majority-class-guessing artifact (a model that always
answers "false" scores accuracy equal to whatever the tier's true
false-rate happens to be, which can look like "improving with difficulty"
if that rate shifts by tier -- fixed by exact 50/50 in every tier, with **no
leakage** miscounted or left out); (2) unequal family sizes, fixed by making
all 5 exactly 20/tier.

Mirrors the structure of the Domain 1 (Geometry) Phase 1 notebook otherwise:
tiered rejection sampling against a named validity rule set, a
compute-invariant-properties → randomize-presentation → serialize →
read-presentation-dependent-property build order, and independent
verification from the serialized string.

**Per the PDF (Section 4.2, Table 6):**

| Tier | Count | Nodes | Purpose |
|------|-------|-------|---------|
| simple | 100 | 6–15  | Baseline; small enough for manual verification |
| medium | 100 | 16–40 | Core measurement |
| hard   | 100 | 41–80 | Stress test |

**8 properties** (Table 8): `degree_of_node_0`, `edge_count` *(local)*,
`triangle_count`, `is_bipartite`, `is_planar`, `diameter`,
`chromatic_number`, `avg_clustering` *(global)*.

Output:
- `graph_exp1_dataset.json` — 300 graphs with edge-list serialization + ground truth
- `graph_exp1_summary.json` — summary statistics (Section 7-equivalent)

Random seed fixed at **42** and recorded in every record.


In [ ]:
# Phase 0: environment
!pip install networkx scipy matplotlib --quiet

import json, math, random, re, time
from collections import deque
import networkx as nx
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt

print("networkx", nx.__version__)

## 1. Tiers and the validity checker

Generation is organized around 4 quadrants on (`is_bipartite`, `is_planar`)
crossed with all 5 of the PDF's Table 7 families, not one family per
quadrant:

| Family | Q1 (bip+planar) | Q2 (bip+¬planar) | Q3 (¬bip+planar) | Q4 (¬bip+¬planar) | Total |
|---|---|---|---|---|---|
| `random_bipartite` | 10 | 10 | — | — | 20 |
| `random_planar` | — | — | 20 | — | 20 |
| `erdos_renyi` | 5 | 5 | 2 | 8 | 20 |
| `barabasi_albert` | 5 | 5 | 2 | 8 | 20 |
| `watts_strogatz` | 5 | 5 | 1 | 9 | 20 |
| **Quadrant total** | **25** | **25** | **25** | **25** | **100** |

`random_bipartite` stays true to its name (always bipartite -- Q1/Q2 only);
`random_planar` stays true to its name (always planar -- Q3 only, no
non-planar instances under that label); `erdos_renyi`/`barabasi_albert`/
`watts_strogatz` cover the rest, each with its own way of hitting every
quadrant (see cell 4/5) -- e.g. some `barabasi_albert` graphs are bipartite,
some `watts_strogatz` graphs are planar, genuinely, not incidentally. Every
family is exactly 20/tier; every quadrant is exactly 25/tier, which is what
makes `is_bipartite` (= Q1+Q2 = 50) and `is_planar` (= Q1+Q3 = 50) exactly
50/50 in every tier, counting every graph regardless of family.

**Validity constraints** (Section 4.2), checked by `check_validity`, each
with a named rejection reason:

| Rule | Rejection reason | Check |
|------|------------------|-------|
| 1 | `node_count_out_of_range` | `vmin <= n <= vmax` |
| 2 | `self_loop` | no self-loops |
| 3 | *(structural)* | no multi-edges -- guaranteed by using `nx.Graph`, never `nx.MultiGraph` |
| 4 | `not_connected` | single connected component |
| 5 | `too_few_edges` | `m >= n - 1` |
| 6 | `too_many_edges` | `m <= C(n,2)/2` |

Every quadrant generator is a **rejection sampler or a construction that
retries on a fresh `n`** when its target combination can't be hit: draw `n`
and family/quadrant-specific parameters, build a candidate, run
`check_validity` plus the quadrant's own boolean checks, retry up to
`max_tries` (300-3000 depending on the generator).


In [ ]:
# Tier node-count ranges (Table 6). No coordinate axis here (unlike geometry) --
# the graph's complexity axis is node count alone.
TIERS = {
    #         vmin vmax
    "simple": (6,  15),
    "medium": (16, 40),
    "hard":   (41, 80),
}

# Per-tier, per-family, per-quadrant plan. All 5 families are exactly
# 20/tier (perfectly equal); each family covers whichever (is_bipartite,
# is_planar) quadrants make sense for it (random_bipartite: only the 2
# bipartite quadrants; random_planar: only its 1 planar-and-not-bipartite
# quadrant; erdos_renyi/barabasi_albert/watts_strogatz: all 4, each with its
# own flavor -- see cell 4/5). Every quadrant totals exactly 25/tier, so
# is_bipartite and is_planar are each exactly 50/50, every tier, counting
# every graph regardless of family (see cell 2/4 for why this design, not
# post-hoc correction, is what's used here).
FAMILY_QUADRANT_PLAN = {
    "random_bipartite": {"Q1": 10, "Q2": 10},
    "random_planar": {"Q3": 20},
    "erdos_renyi": {"Q1": 5, "Q2": 5, "Q3": 2, "Q4": 8},
    "barabasi_albert": {"Q1": 5, "Q2": 5, "Q3": 2, "Q4": 8},
    "watts_strogatz": {"Q1": 5, "Q2": 5, "Q3": 1, "Q4": 9},
}
assert all(sum(q.values()) == 20 for q in FAMILY_QUADRANT_PLAN.values())
assert sum(sum(q.values()) for q in FAMILY_QUADRANT_PLAN.values()) == 100
for quad in ["Q1", "Q2", "Q3", "Q4"]:
    total = sum(q.get(quad, 0) for q in FAMILY_QUADRANT_PLAN.values())
    assert total == 25, (quad, total)


def check_validity(G, vmin, vmax):
    if G is None or G.number_of_nodes() == 0:
        return False, "empty"
    n = G.number_of_nodes()
    if not (vmin <= n <= vmax):                    # rule 1
        return False, "node_count_out_of_range"
    if nx.number_of_selfloops(G) > 0:               # rule 2
        return False, "self_loop"
    if not nx.is_connected(G):                      # rule 4
        return False, "not_connected"
    m = G.number_of_edges()
    if m < n - 1:                                   # rule 5
        return False, "too_few_edges"
    if m > (n * (n - 1) / 2) / 2:                    # rule 6
        return False, "too_many_edges"
    return True, None


print("Tiers, family/quadrant plan, and validity checker defined.")


## 2. The quadrant generators, by family

**Generic (used by `random_bipartite`, `random_planar`, and reused as
`erdos_renyi`'s own Q1/Q2 flavor -- independent edge probability is already
Erdos-Renyi's defining trait, so these don't need a separate ER-specific
version):**

- **`gen_bipartite_planar`** (Q1) -- a bipartite spanning tree with extra
  cross-partition edges added one at a time, kept only if
  `nx.check_planarity` still passes. *Why constructive, not rejection
  sampling:* bipartite planar graphs are capped at `m <= 2n-4` edges, a
  much tighter bound than general planar graphs (`3n-6`), so a randomly
  drawn bipartite graph is planar only by chance, and that chance drops
  fast as `n` grows -- measured before committing to this method: pure
  rejection sampling succeeded only 37/75 times across the 3 tiers, against
  75/75 for construction, and ran ~12x slower even at that reduced rate.
- **`gen_bipartite_nonplanar`** (Q2) -- `nx.bipartite.random_graph(n1, n2,
  p)`, rejection-sampled for `is_planar() == False`.
- **`gen_nonbipartite_planar`** (Q3, `random_planar`'s only quadrant) --
  Delaunay triangulation of random points (always planar), thinned toward a
  target edge count, rejection-sampled for `is_bipartite() == False`.
- **`gen_nonbipartite_nonplanar`** (Q4) -- Erdos-Renyi / Barabasi-Albert /
  Watts-Strogatz at their PDF-specified parameter ranges, rejection-sampled
  for both booleans `False`.

**`erdos_renyi`'s own Q3 flavor** -- `gen_er_nonbip_planar`: a
**uniformly-random-attachment tree** (each new node attaches to a uniformly
random existing node -- the ER-flavored way to grow a tree, as opposed to
preferential attachment) plus one intra-partition edge, kept only if the
graph is still planar. Trees are always bipartite; in a connected bipartite
graph every path between two same-side vertices has even length, so closing
one with an extra edge forces an odd cycle (same trick as the earlier
cross-family leak correction) -- deterministic, no rejection sampling
needed on the property itself.

**`barabasi_albert`'s own quadrant flavors**, all built on preferential
attachment:

- **`gen_ba_bip_planar`** (Q1) -- `m=1` Barabasi-Albert is always a
  **tree**, hence always bipartite and planar, with zero forcing needed.
- **`gen_ba_bip_nonplanar`** (Q2) -- a hub-skewed bipartite graph: a
  spanning tree biased toward a small set of "hub" nodes on each side, then
  extra edges added preferentially to those hubs until non-planar --
  mimicking BA's hub-heavy degree distribution in a genuinely bipartite
  graph (networkx has no native bipartite BA variant).
- **`gen_ba_nonbip_planar`** (Q3) -- a BA tree (`m=1`, preferential
  attachment, unlike `erdos_renyi`'s uniform-attachment tree above) plus
  one intra-partition edge, kept only if still planar.

**`watts_strogatz`'s own quadrant flavors**, all exploiting **cycle
parity**: an even-length cycle is automatically bipartite (alternating
parity) and planar; an odd-length cycle is automatically non-bipartite (one
odd cycle: itself) and planar. WS's own signature move (rewiring) is
layered on top, kept only while the target quadrant still holds:

- **`gen_ws_bip_planar`** (Q1) -- even cycle + a few cross-parity rewires,
  each kept only if planarity survives.
- **`gen_ws_bip_nonplanar`** (Q2) -- even cycle + cross-parity rewires added
  until non-planar.
- **`gen_ws_nonbip_planar`** (Q3) -- a plain odd cycle -- non-bipartite and
  planar with no rewiring needed at all.

All generators pass the shared `rng` (a single `random.Random(seed)`
instance) directly as NetworkX's `seed` parameter -- NetworkX accepts a
`random.Random` instance natively, so one seed stream drives every
generator's parameter draws, every planarity-check retry, and the later
relabeling step, exactly as geometry drives everything from one `rng`.


In [ ]:
def bipartite_split(rng, n):
    # PDF Table 7: n1/n2 (ratio BETWEEN partitions) in [0.3, 0.7], found by
    # exact integer search rather than rounding (see cell 2/4).
    ratio = rng.uniform(0.3, 0.7)
    candidates = []
    for n1 in range(1, n):
        n2 = n - n1
        r = n1 / n2
        if 0.3 <= r <= 0.7:
            candidates.append((abs(r - ratio), n1, n2))
    candidates.sort()
    _, n1, n2 = candidates[0]
    return n1, n2


# ============================================================
# Generic quadrant generators (used directly by random_bipartite,
# random_planar, and reused as erdos_renyi's own Q1/Q2 flavor -- ER's
# defining trait, independent edge probability, is already what these do).
# ============================================================

# ---- Q1: bipartite AND planar (constructive) ----
def gen_bipartite_planar(rng, vmin, vmax, max_tries=300):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        n1, n2 = bipartite_split(rng, n)
        partA, partB = list(range(n1)), list(range(n1, n))

        # bipartite spanning tree: attach each node to a random already-placed
        # node in the OTHER partition.
        nodes_order = [partA[0], partB[0]]
        placed = set(nodes_order)
        remaining = [v for v in range(n) if v not in placed]
        rng.shuffle(remaining)
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edge(nodes_order[0], nodes_order[1])
        for v in remaining:
            other_part = partB if v in set(partA) else partA
            choices = [u for u in other_part if u in placed] or list(placed)
            u = rng.choice(choices)
            G.add_edge(u, v)
            placed.add(v)

        # add extra cross-partition edges one at a time, keep only if planar
        max_planar_edges = 2 * n - 4 if n >= 3 else n - 1
        target_extra = rng.randint(0, max(0, max_planar_edges - (n - 1)))
        possible_extra = [(a, b) for a in partA for b in partB if not G.has_edge(a, b)]
        rng.shuffle(possible_extra)
        added = 0
        for a, b in possible_extra:
            if added >= target_extra:
                break
            G.add_edge(a, b)
            if nx.check_planarity(G)[0]:
                added += 1
            else:
                G.remove_edge(a, b)

        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"n1": n1, "n2": n2, "extra_edges": added}
    return None, None


# ---- Q2: bipartite AND NOT planar ----
def gen_bipartite_nonplanar(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        n1, n2 = bipartite_split(rng, n)
        p = rng.uniform(0.2, 0.45)
        G = nx.bipartite.random_graph(n1, n2, p, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.check_planarity(G)[0]:
            return G, {"n1": n1, "n2": n2, "p": round(p, 4)}
    return None, None


# ---- Q3: NOT bipartite AND planar (Delaunay -- random_planar's own flavor) ----
def gen_nonbipartite_planar(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        pts = [(rng.random(), rng.random()) for _ in range(n)]
        try:
            tri = Delaunay(pts)
        except Exception:
            continue
        edges = set()
        for simplex in tri.simplices:
            for i in range(3):
                for j in range(i + 1, 3):
                    a, b = int(simplex[i]), int(simplex[j])
                    edges.add((min(a, b), max(a, b)))
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edges_from(edges)
        max_planar_edges = 3 * n - 6 if n >= 3 else n - 1
        target_m = rng.randint(n - 1, min(G.number_of_edges(), max_planar_edges))
        el = list(G.edges())
        rng.shuffle(el)
        for e in el:
            if G.number_of_edges() <= target_m:
                break
            G.remove_edge(*e)
            if not nx.is_connected(G):
                G.add_edge(*e)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G):
            return G, {"target_m": target_m}
    return None, None


# ---- Q4: NOT bipartite AND NOT planar (erdos_renyi / barabasi_albert / watts_strogatz) ----
def gen_nonbipartite_nonplanar(rng, vmin, vmax, method, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if method == "erdos_renyi":
            target_degree = rng.uniform(4, 7)
            p = min(max(target_degree / (n - 1), 0.0), 1.0)
            G = nx.gnp_random_graph(n, p, seed=rng)
            params = {"method": method, "p": round(p, 4), "target_degree": round(target_degree, 2)}
        elif method == "barabasi_albert":
            m = rng.choice([3, 4])
            if m >= n:
                continue
            G = nx.barabasi_albert_graph(n, m, seed=rng)
            params = {"method": method, "m": m}
        else:  # watts_strogatz
            k = rng.choice([4, 6])
            if k >= n:
                continue
            p = rng.choice([0.1, 0.3, 0.5])
            G = nx.watts_strogatz_graph(n, k, p, seed=rng)
            params = {"method": method, "k": k, "p": p}
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G) and not nx.check_planarity(G)[0]:
            return G, params
    return None, None


# ============================================================
# erdos_renyi's own Q3 flavor: a uniformly-random-attachment tree (each new
# node attaches to a UNIFORMLY random existing node -- the ER-flavored way
# to grow a tree, as opposed to Barabasi-Albert's preferential attachment)
# plus one intra-partition edge, kept only if the graph is still planar.
# Trees are always bipartite; closing one intra-partition edge forces an
# odd cycle (same trick as the earlier cross-family leak correction).
# ============================================================
def gen_er_nonbip_planar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        G = nx.Graph()
        G.add_node(0)
        for v in range(1, n):
            u = rng.choice(list(G.nodes()))
            G.add_edge(u, v)
        A, B = nx.bipartite.sets(G)
        A, B = list(A), list(B)
        candidates = ([(A[i], A[j]) for i in range(len(A)) for j in range(i + 1, len(A))]
                      + [(B[i], B[j]) for i in range(len(B)) for j in range(i + 1, len(B))])
        rng.shuffle(candidates)
        added = False
        for u, v in candidates:
            G.add_edge(u, v)
            if not nx.is_bipartite(G) and nx.check_planarity(G)[0]:
                added = True
                break
            G.remove_edge(u, v)
        if not added:
            continue
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"method": "erdos_renyi_uniform_tree_plus_edge"}
    return None, None


# ============================================================
# barabasi_albert's own quadrant flavors, all built on its preferential
# attachment construction.
# ============================================================

# Q1: m=1 Barabasi-Albert is always a TREE -- always bipartite AND planar,
# with zero forcing needed.
def gen_ba_bip_planar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        G = nx.barabasi_albert_graph(n, 1, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"method": "barabasi_albert", "m": 1}
    return None, None


# Q2: a hub-skewed bipartite graph -- a spanning tree biased toward a small
# set of "hub" nodes on each side, then extra edges added preferentially to
# those hubs until non-planar. Mimics BA's hub-heavy degree distribution in
# a genuinely bipartite graph (networkx has no native bipartite BA variant).
def gen_ba_bip_nonplanar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        n1, n2 = bipartite_split(rng, n)
        partA, partB = list(range(n1)), list(range(n1, n))
        hubs_a = set(partA[:max(1, n1 // 4)])
        hubs_b = set(partB[:max(1, n2 // 4)])
        nodes_order = [partA[0], partB[0]]
        placed = set(nodes_order)
        remaining = [v for v in range(n) if v not in placed]
        rng.shuffle(remaining)
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edge(nodes_order[0], nodes_order[1])
        for v in remaining:
            other_part = partB if v in set(partA) else partA
            hub_choices = [u for u in other_part if u in placed and (u in hubs_a or u in hubs_b)]
            choices = hub_choices or [u for u in other_part if u in placed] or list(placed)
            u = rng.choice(choices)
            G.add_edge(u, v)
            placed.add(v)
        possible = [(a, b) for a in partA for b in partB if not G.has_edge(a, b)]
        rng.shuffle(possible)
        possible.sort(key=lambda e: 0 if (e[0] in hubs_a or e[1] in hubs_b) else 1)
        for a, b in possible:
            G.add_edge(a, b)
            if not nx.check_planarity(G)[0]:
                break
        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and not nx.check_planarity(G)[0]:
            return G, {"method": "barabasi_albert_hub_skewed"}
    return None, None


# Q3: a BA tree (m=1, preferential attachment -- unlike erdos_renyi's
# uniform-attachment tree above) plus one intra-partition edge, kept only
# if still planar.
def gen_ba_nonbip_planar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        G = nx.barabasi_albert_graph(n, 1, seed=rng)
        A, B = nx.bipartite.sets(G)
        A, B = list(A), list(B)
        candidates = ([(A[i], A[j]) for i in range(len(A)) for j in range(i + 1, len(A))]
                      + [(B[i], B[j]) for i in range(len(B)) for j in range(i + 1, len(B))])
        rng.shuffle(candidates)
        added = False
        for u, v in candidates:
            G.add_edge(u, v)
            if not nx.is_bipartite(G) and nx.check_planarity(G)[0]:
                added = True
                break
            G.remove_edge(u, v)
        if not added:
            continue
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"method": "barabasi_albert_tree_plus_edge"}
    return None, None


# ============================================================
# watts_strogatz's own quadrant flavors, all exploiting cycle parity: an
# EVEN cycle is automatically bipartite (alternating parity) and planar; an
# ODD cycle is automatically non-bipartite (one odd cycle, itself) and
# planar. "Rewiring" (WS's own signature move) is layered on top, kept only
# while the target quadrant still holds.
# ============================================================

# Q1: even cycle + a few cross-parity rewires, kept only while still planar.
def gen_ws_bip_planar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if n % 2 == 1:
            n -= 1
        if n < vmin:
            continue
        G = nx.cycle_graph(n)
        nodes = list(G.nodes())
        rewires = rng.randint(0, max(0, n // 8))
        for _ in range(rewires):
            u, v = rng.sample(nodes, 2)
            if (u - v) % 2 == 0 or G.has_edge(u, v):
                continue
            G.add_edge(u, v)
            if not nx.check_planarity(G)[0]:
                G.remove_edge(u, v)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"method": "watts_strogatz_even_cycle"}
    return None, None


# Q2: even cycle + cross-parity rewires until non-planar.
def gen_ws_bip_nonplanar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if n % 2 == 1:
            n -= 1
        if n < vmin:
            continue
        G = nx.cycle_graph(n)
        nodes = list(G.nodes())
        pairs = [(u, v) for i, u in enumerate(nodes) for v in nodes[i + 1:]
                 if (u - v) % 2 == 1 and not G.has_edge(u, v)]
        rng.shuffle(pairs)
        for u, v in pairs:
            G.add_edge(u, v)
            if not nx.check_planarity(G)[0]:
                break
        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and not nx.check_planarity(G)[0]:
            return G, {"method": "watts_strogatz_even_cycle_rewired"}
    return None, None


# Q3: plain odd cycle -- non-bipartite and planar with no rewiring needed.
def gen_ws_nonbip_planar(rng, vmin, vmax, max_tries=500):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if n % 2 == 0:
            n -= 1
        if n < vmin:
            continue
        G = nx.cycle_graph(n)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"method": "watts_strogatz_odd_cycle"}
    return None, None


print("All quadrant generators defined (generic + erdos_renyi/barabasi_albert/watts_strogatz flavors).")


# ============================================================
# Chromatic-number-exact generators for Q3 (random_planar's bulk) and Q4
# (erdos_renyi/barabasi_albert/watts_strogatz's bulk) -- see cell 4 for the
# balance goal (chromatic_number spread across 3/4/5/6, not just whatever
# falls out naturally). Both use `exact_chromatic_number` (cell 7) to VERIFY
# the target was actually hit, retrying on mismatch rather than trusting
# construction alone -- measured necessary: a naive clique-anchor
# construction occasionally lands one chromatic number higher than intended
# when its "pendant" attachments happen to over-constrain a single node.
# ============================================================

# Q3 flavor (random_planar): reuse the existing Delaunay-based planar
# generator (already reliable and naturally triangle-rich) and rejection
# sample on the exact chromatic number, instead of building a fresh
# clique-anchor construction -- measured necessary: a clique-anchor with any
# bonus triangle-boosting blocks reliably STAYS planar only at zero bonus
# blocks once graphs get large (medium/hard tiers); with any blocks added,
# it almost always develops a K5/K3,3 minor and stops being planar at all.
# Delaunay-derived graphs don't have that fragility and already vary widely
# in triangle count on their own.
def gen_q3_chromatic_exact(rng, vmin, vmax, target_k, max_tries=200):
    for _ in range(max_tries):
        G, _ = gen_nonbipartite_planar(rng, vmin, vmax)
        if G is None:
            continue
        chrom, certified = exact_chromatic_number(G)
        if certified and chrom == target_k:
            return G, {"method": "random_planar_chromatic_exact", "target_k": target_k}
    return None, None


# Q4 flavor (erdos_renyi/barabasi_albert/watts_strogatz): a K_k anchor
# clique (forces clique number, hence chromatic number, to at least k) with
# the rest of the nodes attached as a random tree, plus a random number of
# disjoint bonus k-cliques grafted onto separate tree nodes to vary triangle
# count without changing the chromatic number (a "block graph" of same-size
# cliques glued by bridges has chromatic number = the largest block, a
# classical result) -- verified after construction, not just assumed;
# K5/K6 anchors are automatically non-planar (Kuratowski), so this always
# satisfies Q4's not-planar requirement for free.
def gen_clique_anchor_chromatic(rng, vmin, vmax, k, max_tries=300):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if n < k:
            continue
        G = nx.complete_graph(k)
        remaining = list(range(k, n))
        rng.shuffle(remaining)
        placed = list(range(k))
        for v in remaining:
            u = rng.choice(placed)
            G.add_edge(u, v)
            placed.append(v)

        pendants = [v for v in placed if v >= k]
        rng.shuffle(pendants)
        max_blocks = len(pendants) // k
        block_fraction = rng.choice([0.0, 0.5, 1.0])   # triangle-count diversity
        n_blocks = round(max_blocks * block_fraction)
        for b in range(n_blocks):
            group = pendants[b * k:(b + 1) * k]
            for a in range(len(group)):
                for c in range(a + 1, len(group)):
                    G.add_edge(group[a], group[c])

        ok, _ = check_validity(G, vmin, vmax)
        if not ok or nx.check_planarity(G)[0]:   # Q4 requires NOT planar
            continue
        chrom, certified = exact_chromatic_number(G)
        if certified and chrom == k:
            return G, {"method": "clique_anchor_chromatic_exact", "target_k": k,
                       "n_bonus_blocks": n_blocks}
    return None, None


## 3. Chromatic number

NetworkX has no exact chromatic-number function (Section 4, "Chromatic
number computation"). This implements the PDF's specified fallback chain:

1. **Clique certificate, fast path.** If the greedy `DSATUR` upper bound
   equals the size of a maximum clique found by `nx.find_cliques`, the
   chromatic number is certified immediately — a clique of size *k* forces
   at least *k* colors, and the greedy coloring already achieves *k*.
2. **Exact backtracking, bounded.** Otherwise, search for a valid *k*-coloring
   for increasing *k*, using DSATUR vertex ordering (most-constrained vertex
   first) and color-symmetry breaking (never open a color number more than 1
   past the highest used so far). Time-boxed per graph.
3. **Uncertified fallback.** If the time box is hit before the gap between
   clique lower bound and greedy upper bound closes, the graph's chromatic
   number is **excluded from evaluation** and flagged
   `chromatic_number_certified: false` in its metadata — per the PDF: *"Do
   not use approximate values."*

Verified against known graphs before trusting it on the dataset (execution
checklist item 2: *"verify against known graphs"*).

In [ ]:
def clique_number(G, cap=None):
    best = 1
    for c in nx.find_cliques(G):
        if len(c) > best:
            best = len(c)
        if cap is not None and best >= cap:
            return best
    return best


def greedy_upper_bound(G):
    coloring = nx.coloring.greedy_color(G, strategy="DSATUR")
    return max(coloring.values()) + 1 if coloring else 1


def exact_chromatic_number(G, time_limit=15.0):
    # Return (k, certified). Exact via DSATUR-ordered, symmetry-broken
    # backtracking within time_limit; else an uncertified greedy upper bound.
    n = G.number_of_nodes()
    if n == 0:
        return 0, True
    ub = greedy_upper_bound(G)
    lb = clique_number(G, cap=ub)
    if lb == ub:
        return ub, True                      # clique certificate

    nodes = list(G.nodes())
    adj = {v: set(G.neighbors(v)) for v in nodes}
    deadline = time.monotonic() + time_limit
    timed_out = [False]

    def can_color(k):
        colors = {}
        steps = [0]

        def choose_next():
            # DSATUR: most saturated (distinct neighbor colors) first, ties by degree.
            best, best_sat, best_deg = None, -1, -1
            for v in nodes:
                if v in colors:
                    continue
                sat = len({colors[u] for u in adj[v] if u in colors})
                deg = len(adj[v])
                if sat > best_sat or (sat == best_sat and deg > best_deg):
                    best, best_sat, best_deg = v, sat, deg
            return best

        def backtrack(count):
            steps[0] += 1
            if steps[0] % 1000 == 0 and time.monotonic() > deadline:
                timed_out[0] = True
                return False
            if count == n:
                return True
            v = choose_next()
            used = {colors[u] for u in adj[v] if u in colors}
            max_used = max(colors.values(), default=-1)
            upper = min(k - 1, max_used + 1)   # color-symmetry breaking
            for c in range(upper + 1):
                if c not in used:
                    colors[v] = c
                    if backtrack(count + 1):
                        return True
                    del colors[v]
                    if timed_out[0]:
                        return False
            return False

        return backtrack(0)

    for k in range(lb, ub + 1):
        if can_color(k):
            return k, True
        if timed_out[0]:
            break
    return ub, False                          # uncertified: exclude at evaluation time


# Known-graph sanity check (execution checklist item 2).
_known = [
    ("K5", nx.complete_graph(5), 5), ("Petersen", nx.petersen_graph(), 3),
    ("C5", nx.cycle_graph(5), 3), ("C6", nx.cycle_graph(6), 2),
    ("K3,3", nx.complete_bipartite_graph(3, 3), 2),
    ("K4", nx.complete_graph(4), 4), ("Star_10", nx.star_graph(10), 2),
]
for name, G, expected in _known:
    k, certified = exact_chromatic_number(G)
    assert k == expected, f"{name}: got {k}, expected {expected}"
    print(f"  {name}: chromatic_number={k} (certified={certified}) -- matches expected {expected}")
print("Chromatic number verified against", len(_known), "known graphs.")

## 4. Ground truth (8 properties) + record builder

**The step order here is a correctness requirement**, same principle as the
geometry domain's winding-reversal ordering (Section 3.2 there):

1. **`compute_presentation_independent(G)`** — the 6 properties that do not
   depend on node labeling: `triangle_count`, `is_bipartite`, `is_planar`,
   `diameter`, `chromatic_number`, `avg_clustering`.
2. **`randomize_labeling(G, rng)`** — relabel nodes with a random permutation
   of `0..n-1`. Without this, node `0` is whatever label the generator
   happened to assign — e.g. Barabási–Albert's earliest nodes are
   structurally the hubs — which would make `degree_of_node_0` a function of
   generator internals rather than a genuine "read this from the
   serialization" question. This is the graph-domain analogue of the
   geometry domain's `maybe_reverse` (winding direction): any property whose
   ground truth depends on how the object is written down must be computed
   *after* randomizing that presentation.
3. **`to_edge_list_string(G)`** — serialize the *relabeled* graph, edges
   sorted `(min(u,v), max(u,v))` then lexicographically, per Section 4.3.
4. **`degree_of_node_0`, `edge_count`** — read from the relabeled graph, so
   the label matches the string the model will actually see.

All floats are rounded to 4 decimal places.

In [ ]:
# Step 1: properties that do NOT depend on node labeling.
def compute_presentation_independent(G):
    triangle_count = sum(nx.triangles(G).values()) // 3
    is_bipartite = bool(nx.is_bipartite(G))
    is_planar = bool(nx.check_planarity(G)[0])
    diameter = nx.diameter(G)
    avg_clustering = round(nx.average_clustering(G), 4)
    clique_n = clique_number(G)
    chrom, certified = exact_chromatic_number(G)
    props = {
        "triangle_count": triangle_count,
        "is_bipartite": is_bipartite,
        "is_planar": is_planar,
        "diameter": diameter,
        "chromatic_number": chrom,
        "avg_clustering": avg_clustering,
    }
    extra = {"clique_number": clique_n, "chromatic_number_certified": certified}
    return props, extra


# Step 2: relabel with a random permutation of 0..n-1 (see markdown above).
def randomize_labeling(G, rng):
    nodes = list(G.nodes())
    shuffled = nodes[:]
    rng.shuffle(shuffled)
    mapping = {old: new for old, new in zip(nodes, shuffled)}
    return nx.relabel_nodes(G, mapping, copy=True)


# Step 3: serialize to the edge-list format (Section 4.3).
def to_edge_list_string(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    edges = sorted((min(u, v), max(u, v)) for u, v in G.edges())
    lines = [f"GRAPH (n={n}, m={m}):"] + [f"{u} {v}" for u, v in edges]
    return "\n".join(lines)


# Build one full dataset record, enforcing the correct step order.
def build_record(G, tier, family, index, rng, seed, gen_params):
    props, extra = compute_presentation_independent(G)   # step 1
    G2 = randomize_labeling(G, rng)                       # step 2
    edge_list = to_edge_list_string(G2)                   # step 3
    props["degree_of_node_0"] = G2.degree(0)              # step 4
    props["edge_count"] = G2.number_of_edges()

    return {
        "object_id": f"graph_{tier}_{family}_{index:03d}",
        "tier": tier,
        "family": family,
        "num_nodes": G2.number_of_nodes(),
        "num_edges": G2.number_of_edges(),
        "edge_list": edge_list,
        "properties": props,
        "metadata": {
            "generation_params": gen_params,
            "random_seed": seed,
            "clique_number": extra["clique_number"],
            "chromatic_number_certified": extra["chromatic_number_certified"],
            "is_connected": True,
        },
    }


print("Ground-truth and record builder defined (8 properties).")

## 5. Build the 300-graph dataset + summary

300 graphs = 3 tiers x 100. Within each tier, `FAMILY_QUADRANT_PLAN` (cell
3) drives generation: 5 families x 20/tier, each split across whichever
quadrants it covers via `QUADRANT_GENERATORS` (cell 11). Every quadrant
totals exactly 25/tier, giving exact 50/50 on both `is_bipartite` and
`is_planar` -- verified in `boolean_balance_by_tier` below by counting
every graph regardless of family, not just within any one "designed"
family. Generation failure is **fatal**: if a generator exhausts its retry
budget, `build_dataset` raises rather than emitting a short dataset.


In [ ]:
# Which generator function builds each family's each quadrant. A quadrant
# entry can be a single function (called for all of that quadrant's slots)
# or a list of (count, function) sub-buckets, used where a quadrant needs
# more than one chromatic-number target (see cell 4): random_planar's Q3
# splits 7/13 between chromatic 3 and 4 (topping up what
# erdos_renyi/barabasi_albert/watts_strogatz's own Q3 flavor -- always
# chromatic 3 -- doesn't cover); erdos_renyi/barabasi_albert/watts_strogatz's
# Q4 splits between chromatic 5 and 6 (4/4, 4/4, 4/5).
QUADRANT_GENERATORS = {
    "random_bipartite": {"Q1": gen_bipartite_planar, "Q2": gen_bipartite_nonplanar},
    "random_planar": {
        "Q3": [
            (7, lambda rng, vmin, vmax: gen_q3_chromatic_exact(rng, vmin, vmax, 3)),
            (13, lambda rng, vmin, vmax: gen_q3_chromatic_exact(rng, vmin, vmax, 4)),
        ],
    },
    "erdos_renyi": {
        "Q1": gen_bipartite_planar, "Q2": gen_bipartite_nonplanar,
        "Q3": gen_er_nonbip_planar,
        "Q4": [
            (4, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 5)),
            (4, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 6)),
        ],
    },
    "barabasi_albert": {
        "Q1": gen_ba_bip_planar, "Q2": gen_ba_bip_nonplanar,
        "Q3": gen_ba_nonbip_planar,
        "Q4": [
            (4, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 5)),
            (4, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 6)),
        ],
    },
    "watts_strogatz": {
        "Q1": gen_ws_bip_planar, "Q2": gen_ws_bip_nonplanar,
        "Q3": gen_ws_nonbip_planar,
        "Q4": [
            (4, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 5)),
            (5, lambda rng, vmin, vmax: gen_clique_anchor_chromatic(rng, vmin, vmax, 6)),
        ],
    },
}

# Draw order within each family, so indices are contiguous and predictable:
# graph_{tier}_{family}_001.. is always Q1 first, then Q2, Q3, Q4 (whichever
# the family covers, per FAMILY_QUADRANT_PLAN).
QUADRANT_ORDER = ["Q1", "Q2", "Q3", "Q4"]


def build_dataset(seed=42):
    rng = random.Random(seed)
    records = []
    for tier in ["simple", "medium", "hard"]:
        vmin, vmax = TIERS[tier]
        for family, quad_counts in FAMILY_QUADRANT_PLAN.items():
            index = 1
            for quad in QUADRANT_ORDER:
                if quad not in quad_counts:
                    continue
                gen_spec = QUADRANT_GENERATORS[family][quad]
                sub_buckets = gen_spec if isinstance(gen_spec, list) else [(quad_counts[quad], gen_spec)]
                assert sum(c for c, _ in sub_buckets) == quad_counts[quad]
                for count, gen in sub_buckets:
                    for _ in range(count):
                        G, params = gen(rng, vmin, vmax)
                        if G is None:
                            raise RuntimeError(f"Failed to generate {tier}/{family}/{quad} #{index}")
                        records.append(build_record(G, tier, family, index, rng, seed, params))
                        index += 1
    return records


# Min/max/mean/median/std for a list of numbers (population std, same as geometry).
def stats_for(values):
    n = len(values)
    mean = sum(values) / n
    sv = sorted(values)
    median = sv[n // 2] if n % 2 else (sv[n // 2 - 1] + sv[n // 2]) / 2
    var = sum((v - mean) ** 2 for v in values) / n
    return {"min": round(min(values), 2), "max": round(max(values), 2),
            "mean": round(mean, 2), "median": round(median, 2),
            "std": round(var ** 0.5, 2)}


def summarize(records):
    summary = {"total": len(records)}
    by_tier = {}
    for r in records:
        by_tier.setdefault(r["tier"], {}).setdefault(r["family"], 0)
        by_tier[r["tier"]][r["family"]] += 1
    summary["counts_by_tier_family"] = by_tier

    n_bip = sum(1 for r in records if r["properties"]["is_bipartite"])
    n_planar = sum(1 for r in records if r["properties"]["is_planar"])
    summary["bipartite_overall"] = n_bip
    summary["planar_overall"] = n_planar

    # Per-tier true/false counts for both booleans, counting every graph
    # regardless of family (the direct check that there is exact 50/50
    # balance -- see cell 2/4), plus a per-family-per-quadrant breakdown to
    # show the cross-family overlap (e.g. some erdos_renyi graphs are
    # bipartite or planar) is real and correctly sized.
    boolean_balance = {}
    quadrant_by_family = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        boolean_balance[tier] = {
            "is_bipartite": {
                "true": sum(1 for r in rs if r["properties"]["is_bipartite"]),
                "false": sum(1 for r in rs if not r["properties"]["is_bipartite"]),
            },
            "is_planar": {
                "true": sum(1 for r in rs if r["properties"]["is_planar"]),
                "false": sum(1 for r in rs if not r["properties"]["is_planar"]),
            },
        }
        qbf = {}
        for r in rs:
            key = r["family"]
            qbf.setdefault(key, {"bip_true_plan_true": 0, "bip_true_plan_false": 0,
                                  "bip_false_plan_true": 0, "bip_false_plan_false": 0})
            b = r["properties"]["is_bipartite"]
            p = r["properties"]["is_planar"]
            qbf[key][f"bip_{b}_plan_{p}".replace("True", "true").replace("False", "false")] += 1
        quadrant_by_family[tier] = qbf
    summary["boolean_balance_by_tier"] = boolean_balance
    summary["quadrant_by_family_by_tier"] = quadrant_by_family

    dist = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        dist[tier] = {
            "num_nodes": stats_for([r["num_nodes"] for r in rs]),
            "num_edges": stats_for([r["num_edges"] for r in rs]),
            "triangle_count": stats_for([r["properties"]["triangle_count"] for r in rs]),
            "diameter": stats_for([r["properties"]["diameter"] for r in rs]),
            "chromatic_number": stats_for([r["properties"]["chromatic_number"] for r in rs]),
            "avg_clustering": stats_for([r["properties"]["avg_clustering"] for r in rs]),
            "edge_list_length": stats_for([len(r["edge_list"]) for r in rs]),
        }
    summary["distribution_by_tier"] = dist

    uncertified = [r["object_id"] for r in records if not r["metadata"]["chromatic_number_certified"]]
    summary["chromatic_number_uncertified_count"] = len(uncertified)
    summary["chromatic_number_uncertified_ids"] = uncertified
    return summary


# Run it all.
SEED = 42
print("Generating 300-graph dataset (seed =", SEED, ")...")
t0 = time.time()
records = build_dataset(SEED)
print(f"Done in {time.time() - t0:.1f}s.")

with open("graph_exp1_dataset.json", "w") as f:
    json.dump(records, f, indent=2)
print("Saved graph_exp1_dataset.json with", len(records), "graphs.")

summary = summarize(records)
with open("graph_exp1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved graph_exp1_summary.json.\n")

print("Total graphs:", summary["total"])
print("Counts by tier/family:")
for tier, fams in summary["counts_by_tier_family"].items():
    print(f"  {tier}: {fams}")
print(f"\nBipartite overall: {summary['bipartite_overall']}/300"
      f" ({100*summary['bipartite_overall']/300:.1f}%)")
print(f"Planar overall:    {summary['planar_overall']}/300"
      f" ({100*summary['planar_overall']/300:.1f}%)")

print("\nBoolean balance by tier (should be exactly 50/50 on both, every tier):")
for tier, props in summary["boolean_balance_by_tier"].items():
    b = props["is_bipartite"]
    p = props["is_planar"]
    print(f"  {tier}: is_bipartite true={b['true']} false={b['false']} | "
          f"is_planar true={p['true']} false={p['false']}")

print("\nCross-family overlap check (simple tier, bip/planar counts per family):")
for fam, counts in summary["quadrant_by_family_by_tier"]["simple"].items():
    print(f"  {fam}: {counts}")

print("\nChromatic number distribution by tier (2 is the locked bipartite half; "
      "3/4/5/6 should be roughly even):")
for tier in ["simple", "medium", "hard"]:
    from collections import Counter
    vals = Counter(r["properties"]["chromatic_number"] for r in records if r["tier"] == tier)
    print(f"  {tier}: {dict(sorted(vals.items()))}")

print("\nTriangle count spread among non-bipartite graphs, by tier (min/median/max, "
      "checking it's not collapsed onto one value):")
for tier in ["simple", "medium", "hard"]:
    vals = sorted(r["properties"]["triangle_count"] for r in records
                  if r["tier"] == tier and not r["properties"]["is_bipartite"])
    n = len(vals)
    print(f"  {tier}: min={vals[0]} median={vals[n//2]} max={vals[-1]}")

print(f"\nChromatic number uncertified: {summary['chromatic_number_uncertified_count']}"
      f" {summary['chromatic_number_uncertified_ids']}")
print("\nNode-count range per tier:")
for tier, d in summary["distribution_by_tier"].items():
    nc = d["num_nodes"]
    print(f"  {tier}: min {nc['min']}, max {nc['max']}, mean {nc['mean']}")


## 6. Independent verification

Re-parse each record **from the stored edge-list string** with
`parse_edge_list` (plain text parsing, no networkx) and recompute properties
with hand-written implementations — BFS-based bipartiteness and diameter,
adjacency-set triangle counting, hand-rolled clustering coefficient — then
compare against the stored ground truth.

**Known gap** (documented, same discipline as the geometry domain's own
gap): this covers 6 of 8 properties. `is_planar` and `chromatic_number` are
not independently re-derived — an independent planarity test (Boyer–Myrvold)
and an independent exact-coloring implementation are both substantial
undertakings on their own; worth closing in a later pass, same as geometry's
unclosed `bbox`/`centroid`/`convex`/`orientation` gap.

In [ ]:
def parse_edge_list(text):
    lines = text.strip().split("\n")
    m = re.match(r"GRAPH \(n=(\d+), m=(\d+)\):", lines[0])
    n_hdr, m_hdr = int(m.group(1)), int(m.group(2))
    edges = []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        u, v = map(int, line.split())
        edges.append((u, v))
    return n_hdr, m_hdr, edges


def indep_degree0(edges):
    return sum(1 for u, v in edges if u == 0 or v == 0)


def indep_triangle_count(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    count = 0
    for u in range(n):
        for v in adj[u]:
            if v > u:
                count += sum(1 for w in (adj[u] & adj[v]) if w > v)
    return count


def indep_is_bipartite(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    color = [-1] * n
    for start in range(n):
        if color[start] != -1:
            continue
        color[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if color[w] == -1:
                    color[w] = 1 - color[u]
                    q.append(w)
                elif color[w] == color[u]:
                    return False
    return True


def indep_diameter(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    diam = 0
    for start in range(n):
        dist = [-1] * n
        dist[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if dist[w] == -1:
                    dist[w] = dist[u] + 1
                    q.append(w)
        if -1 in dist:
            return None    # would indicate a disconnected graph -- shouldn't happen
        diam = max(diam, max(dist))
    return diam


def indep_avg_clustering(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    total = 0.0
    for u in range(n):
        nb = list(adj[u])
        k = len(nb)
        if k < 2:
            continue
        links = 0
        for i in range(len(nb)):
            for j in range(i + 1, len(nb)):
                if nb[j] in adj[nb[i]]:
                    links += 1
        total += (2 * links) / (k * (k - 1))
    return total / n


mism = 0
for r in records:
    n, m_hdr, edges = parse_edge_list(r["edge_list"])
    gt = r["properties"]
    checks = {
        "num_nodes_header": (n, r["num_nodes"], 0),
        "edge_count_header": (m_hdr, gt["edge_count"], 0),
        "degree_of_node_0": (indep_degree0(edges), gt["degree_of_node_0"], 0),
        "edge_count": (len(edges), gt["edge_count"], 0),
        "triangle_count": (indep_triangle_count(n, edges), gt["triangle_count"], 0),
        "is_bipartite": (indep_is_bipartite(n, edges), gt["is_bipartite"], 0),
        "diameter": (indep_diameter(n, edges), gt["diameter"], 0),
        "avg_clustering": (indep_avg_clustering(n, edges), gt["avg_clustering"], 1e-4),
    }
    for name, (got, exp, tol) in checks.items():
        ok = (got == exp) if isinstance(exp, bool) else (abs(got - exp) <= tol)
        if not ok:
            mism += 1
            if mism <= 10:
                print(f"MISMATCH {r['object_id']} {name}: indep={got} stored={exp}")

print(f"\nIndependent verification: {mism} mismatches across {len(records)} graphs.")

# connectivity / structural sanity
bad = [r["object_id"] for r in records if not r["metadata"]["is_connected"]]
print("Disconnected records:", len(bad))
print("Properties present:", list(records[0]["properties"].keys()))

## 7. Visual spot-check

Draw one graph per (tier x column) -- 8 columns x 3 tiers = 24 graphs --
for an eyeball check. Each of the 5 families gets one column per quadrant it
covers (`random_bipartite`: Q1+Q2; `random_planar`: Q3 only;
`erdos_renyi`/`barabasi_albert`/`watts_strogatz`: their Q1 slice, to make
the cross-family bipartite/planar overlap visible). Titles show each
graph's actual `is_bipartite`/`is_planar` values.


In [ ]:
SPOTCHECK_COLUMNS = [
    ("random_bipartite (Q1: bip+planar)", "random_bipartite", 1),
    ("random_bipartite (Q2: bip+nonplanar)", "random_bipartite", 11),
    ("random_planar (Q3)", "random_planar", 1),
    ("erdos_renyi (Q1: bip+planar)", "erdos_renyi", 1),
    ("barabasi_albert (Q1: bip+planar)", "barabasi_albert", 1),
    ("watts_strogatz (Q1: bip+planar)", "watts_strogatz", 1),
    ("erdos_renyi (Q4: natural)", "erdos_renyi", 13),
    ("watts_strogatz (Q3: nonbip+planar)", "watts_strogatz", 11),
]


def find_by_index(tier, family, index):
    target_id = f"graph_{tier}_{family}_{index:03d}"
    for r in records:
        if r["object_id"] == target_id:
            return r
    return None

fig, axes = plt.subplots(3, 8, figsize=(30, 12))
for row, tier in enumerate(["simple", "medium", "hard"]):
    for col, (label, family, index) in enumerate(SPOTCHECK_COLUMNS):
        ax = axes[row][col]
        r = find_by_index(tier, family, index)
        n, m, edges = parse_edge_list(r["edge_list"])
        G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
        pos = nx.spring_layout(G, seed=42)
        nx.draw(G, pos, ax=ax, node_size=25, width=0.6, node_color="#4C72B0")
        ax.set_title(f"{tier}/{label}\nn={n} m={m} bip={r['properties']['is_bipartite']} "
                     f"plan={r['properties']['is_planar']}", fontsize=7)
plt.tight_layout()
plt.savefig("spotcheck_exp1_graph.png", dpi=110)
plt.show()
print("Saved spotcheck_exp1_graph.png")


## 8. Download (Colab)

Download the dataset, summary, and spot-check figure. (Skip if running
locally — the files are already saved in the working directory.)

In [ ]:
try:
    from google.colab import files
    files.download("graph_exp1_dataset.json")
    files.download("graph_exp1_summary.json")
    files.download("spotcheck_exp1_graph.png")
except Exception as e:
    print("Not on Colab (files saved locally):", e)